# Day 15 Tutorial：用 NumPy 理解张量与 shape

## Goal

读懂标量、向量、矩阵与三维数组，预测矩阵乘法和广播 shape，并识别 `(n,)` 与 `(n,1)` 混用造成的危险广播。


## Setup

只依赖 NumPy 和 pandas；不导入 PyTorch，也不训练模型。


In [1]:
import platform
import numpy as np
import pandas as pd

print({"python": platform.python_version(), "numpy": np.__version__})


{'python': '3.10.20', 'numpy': '2.2.6'}


## Steps

### 1. 从 `shape` 和 `ndim` 认识数组


In [2]:
examples = {
    "scalar": np.array(3.0),
    "vector": np.array([1.0, 2.0, 3.0, 4.0]),
    "matrix": np.zeros((4, 3)),
    "three_dimensional": np.zeros((2, 4, 3)),
}
shape_table = pd.DataFrame([
    {"name": name, "shape": str(value.shape), "ndim": value.ndim, "size": value.size}
    for name, value in examples.items()
])
display(shape_table)


,name,shape,ndim,size
0,scalar,(),0,1
1,vector,"(4,)",1,4
2,matrix,"(4, 3)",2,12
3,three_dimensional,"(2, 4, 3)",3,24


### 2. 追踪输入、权重、偏置和输出


In [3]:
X = np.array([
    [1.0, 0.0, 2.0],
    [0.5, 1.0, 1.5],
    [2.0, 1.0, 0.0],
    [1.5, 0.5, 1.0],
])
y = np.array([1.2, 0.8, 1.5, 1.1])
W = np.array([
    [0.2, -0.1],
    [0.4, 0.3],
    [-0.2, 0.5],
])
b = np.array([0.1, -0.1])

hidden_linear = X @ W
hidden_with_bias = hidden_linear + b
y_column = y.reshape(-1, 1)

shape_map = pd.DataFrame([
    {"name": "X", "shape": str(X.shape), "meaning": "samples × input features"},
    {"name": "y", "shape": str(y.shape), "meaning": "one target per sample"},
    {"name": "W", "shape": str(W.shape), "meaning": "input features × hidden units"},
    {"name": "b", "shape": str(b.shape), "meaning": "one bias per hidden unit"},
    {"name": "X @ W + b", "shape": str(hidden_with_bias.shape), "meaning": "samples × hidden units"},
    {"name": "y_column", "shape": str(y_column.shape), "meaning": "samples × one output"},
])
display(shape_map)


,name,shape,meaning
0,X,"(4, 3)",samples × input features
1,y,"(4,)",one target per sample
2,W,"(3, 2)",input features × hidden units
3,b,"(2,)",one bias per hidden unit
4,X @ W + b,"(4, 2)",samples × hidden units
5,y_column,"(4, 1)",samples × one output


### 3. 观察危险广播

列预测 `(4,1)` 与一维标签 `(4,)` 相减不会报错，而会得到 `(4,4)`。


In [4]:
prediction_column = np.arange(4, dtype=float).reshape(-1, 1)
dangerous_error_matrix = prediction_column - y
correct_error_column = prediction_column - y_column

print({
    "prediction_column": prediction_column.shape,
    "y_1d": y.shape,
    "dangerous_result": dangerous_error_matrix.shape,
    "correct_result": correct_error_column.shape,
})


{'prediction_column': (4, 1), 'y_1d': (4,), 'dangerous_result': (4, 4), 'correct_result': (4, 1)}


### 4. 故意制造不匹配并阅读预期异常


In [5]:
bad_W = np.zeros((4, 2))
try:
    X @ bad_W
except ValueError as error:
    expected_error_type = type(error).__name__
    expected_error_message = str(error)
    print("Expected shape error:", expected_error_type)
    print(expected_error_message[:180])


Expected shape error: ValueError
matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 4 is different from 3)


## Checks

把 shape 预期变成断言。


In [6]:
assert X.ndim == 2 and y.ndim == 1
assert X.shape == (4, 3)
assert X.shape[0] == y.shape[0]
assert X.shape[1] == W.shape[0]
assert hidden_linear.shape == (4, 2)
assert hidden_with_bias.shape == (4, 2)
assert y_column.shape == (4, 1)
assert dangerous_error_matrix.shape == (4, 4)
assert correct_error_column.shape == (4, 1)
assert expected_error_type == "ValueError"

print("Checks passed: matrix multiplication, broadcasting, and target shapes are understood.")


Checks passed: matrix multiplication, broadcasting, and target shapes are understood.


## Next Steps

先完成 shape 练习，再进入 Day 16 的 MLP 前向传播。记住 `(902,)` 是长度为 902 的一维 shape；逗号是单元素元组语法，不表示缺失列。shape 正确仍需检查行对齐、单位和轴语义。
